In [ ]:
import pandas as pd
import numpy as np

# === Origen ===
df = pd.read_excel("C:\\Users\\User\\Downloads\\MAYOR Y BALANCE JUNIO 25.xlsx", sheet_name="Report")

df.head()

In [ ]:
# === Tipos iniciales (equivalente a TransformColumnTypes) ===
df = df.astype({
    "CP SUPREMO SAS": "string",
    "Unnamed: 1": "string",
    "Unnamed: 2": "string",
    "MAYOR Y BALANCES CON TERCEROS": "string",
    "Unnamed: 11": "string"
    # el resto lo dejamos como object/float
})


In [ ]:
# === Renombrar columnas (Columnas con nombre cambiado2) ===
df = df.rename(columns={
    "CP SUPREMO SAS": " Cuenta",
    "Unnamed: 1": "Column2",
    "Unnamed: 2": " Descripcion",
    "Unnamed: 3": "Column4",
    "Unnamed: 4": "Column5",
    "Unnamed: 5": "Column6",
    "Unnamed: 6": "Column7",
    "MAYOR Y BALANCES CON TERCEROS": "Column8",
    "Unnamed: 8": "Saldo anterior",
    "Unnamed: 9": "Column10",
    "Unnamed: 10": "Debitos ",
    "Unnamed: 11": "Column12",
    "Unnamed: 12": "Creditos ",
    "Unnamed: 13": "Column14",
    "Unnamed: 14": "Column15",
    "Unnamed: 15": "Saldo",
})

In [ ]:
# === Quitar primeras 5 filas ===
df = df.iloc[6:].reset_index(drop=True)

In [ ]:
# === Quitar filas totalmente vacías ===
df = df.dropna(how="all")

In [ ]:
# === Renombrar columnas finales (como en M) ===
df = df.rename(columns={
    " Cuenta": "PUC",
    "Saldo anterior": "Saldo inicial",
    "Saldo": "Saldo final"
})

In [ ]:
# === Reordenar columnas ===
df = df[["PUC","Column2"," Descripcion","Saldo inicial","Debitos ","Creditos ","Saldo final"]]

In [ ]:
####################### === PUC como número, quitar errores ===
# df["PUC"] = pd.to_numeric(df["PUC"], errors="coerce")
# df = df.dropna(subset=["PUC"])

# Paso 1: Identificar las filas nulas originales
original_nulls = df["PUC"].isna()

# Paso 2: Intentar convertir a número, usando 'coerce' para que los errores sean NaN
df["PUC"] = pd.to_numeric(df["PUC"], errors='coerce')

# Paso 3: Conservar filas si:
# - El valor convertido NO es NaN (es decir, la conversión fue exitosa), o
# - El valor original ya era nulo (isna)
df_limpio = df[~df["PUC"].isna() | original_nulls].copy()

df.head(205)

In [ ]:
# === Saldo inicial, Débitos, Créditos, Saldo final como numéricos, quitar errores ===
# for col in ["Saldo inicial", "Debitos ", "Creditos ", "Saldo final"]:
#    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
df = df.dropna(subset=["Saldo inicial"])   # igual que Table.RemoveRowsWithErrors({"Saldo inicial"})

In [ ]:
# === Columna Personalizado ===
df["Personalizado"] = np.where(df["Saldo inicial"].isna(), df[" Descripcion"], "")

df.head(25)

In [ ]:
# === Desplazar columna Personalizado (shift) ===
df["Personalizado"] = df["Personalizado"].shift(-1)

In [ ]:
# === Combinar columnas ===
df["Combinada"] = (df[" Descripcion"].fillna("") + " " + df["Personalizado"].fillna("")).str.strip()

In [ ]:
# === Filtrar filas con saldo inicial !== null ===
df = df[df["Saldo inicial"].notna()]

In [ ]:
# === Quitar columnas innecesarias ===
df = df.drop(columns=[
    "Column4","Column5","Column6","Column7","Column8",
    "Column10","Column12","Column14","Column15","Column17"
], errors="ignore")

In [ ]:
# === Guardar copia de Column2 ANTES de modificarla ===
df["Col2_original"] = df["Column2"]

In [ ]:
# === Poner capitalizado ===
df["Combinada"] = df["Combinada"].str.title()

In [ ]:
# === Columna Tercero ===
# === Poner en mayúsculas cada palabra ===
df["Combinada"] = df["Combinada"].str.title()

In [ ]:
df.head(25)

In [ ]:
# === Columna Tercero (equivalente a "Personalizada agregada") ===
df["Tercero"] = np.where(df["Col2_original"].notna(), df["Combinada"], "")

In [ ]:
df.head(25)

In [ ]:
# === Quitar Column2 y ajustar PUC a string ===
df = df.drop(columns=["Column2"])
df["PUC"] = df["PUC"].astype(str)

In [ ]:
# === Personalizado concatenado con 01 y FillDown ===
df["Personalizado"] = (df["PUC"] + "01").replace("nan01", np.nan)
df["Personalizado"] = df["Personalizado"].ffill()   # sin warning

In [ ]:
# === Personalizado.1 ===
df["Personalizado.1"] = np.where(df["Tercero"]=="", df["PUC"], df["Personalizado"])

In [ ]:
df.head(25)

In [ ]:
# === Reemplazar PUC final ===
df = df.drop(columns=["PUC","Personalizado"])
df = df.rename(columns={"Personalizado.1":"PUC","Combinada":"Descripción"})

In [ ]:
# === Columna Cuenta ===
df["Cuenta"] = df["PUC"].str[0].map({
    "1":"Activo","2":"Pasivo","3":"Patrimonio",
    "4":"Ingreso","5":"Gasto"
}).fillna("Costo")

In [ ]:
# === Largo de PUC ===
df["Largo"] = df["PUC"].str.len()

In [ ]:
# === Columna Dato Financiero ===
df["Dato Financiero"] = np.select(
    [
        df["PUC"].str.startswith("1"),
        df["PUC"].str.startswith("2"),
        df["PUC"].str.startswith("3"),
        df["PUC"].str.startswith("4"),
    ],
    [
        df["Saldo final"],
        -df["Saldo final"],
        -df["Saldo final"],
        df["Creditos "] - df["Debitos "],
    ],
    default=df["Debitos "] - df["Creditos "]
)


In [ ]:
# === Reorden final ===
df = df[[
    "Largo","Cuenta","PUC","Descripción","Tercero",
    "Saldo inicial","Debitos ","Creditos ","Saldo final","Dato Financiero"
]]

In [ ]:
df.head()

In [ ]:
# Exportar a Excel
#df.to_excel(
#    r"C:\Users\User\Downloads\limpiado_CP.xlsx",  # Ruta de salida
#    index=False                                      # No guardar índice
#)